In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns 

In [2]:
df = pd.read_csv("Ipl_Prediction")
df.sample(5)

,batting_team,bowling_team,city,runs_left,balls_left,wickets,total_runs_x,crr,rrr,result
17072,Sunrisers Hyderabad,Kolkata Knight Riders,Hyderabad,22,27,6,130,6.967742,4.888889,1
48687,Delhi Daredevils,Kolkata Knight Riders,Kolkata,36,20,5,181,8.700000,10.800000,0
45516,Royal Challengers Bangalore,Mumbai Indians,Bengaluru,137,85,9,197,10.285714,9.670588,0
57787,Mumbai Indians,Kolkata Knight Riders,Mumbai,133,114,10,147,14.000000,7.000000,1
29368,Kings XI Punjab,Royal Challengers Bangalore,Durban,62,33,8,168,7.310345,11.272727,1


In [3]:
df.describe()

,runs_left,balls_left,wickets,total_runs_x,crr,rrr,result
count,72413.000000,72413.000000,72413.000000,72413.000000,72413.000000,7.241300e+04,72413.000000
mean,92.258379,62.657078,7.535967,165.583956,7.439523,3.869481e+06,0.525624
std,50.021962,33.404593,2.138637,29.282200,2.275850,9.147546e+07,0.499346
min,-16.000000,-2.000000,0.000000,65.000000,0.000000,-4.200000e+08,0.000000
25%,53.000000,35.000000,6.000000,146.000000,6.257143,7.140000e+00,0.000000
50%,92.000000,63.000000,8.000000,165.000000,7.480519,8.877551e+00,1.000000
75%,130.000000,92.000000,9.000000,185.000000,8.682353,1.090909e+01,1.000000
max,249.000000,119.000000,10.000000,250.000000,42.000000,5.820000e+09,1.000000


In [4]:
df.dropna(inplace = True)

In [5]:
from sklearn.model_selection import train_test_split
x = df.iloc[:,:-1]
y = df.iloc[:,-1]

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.2,random_state = 42)

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import PowerTransformer

trf = ColumnTransformer([
    ("trf1",OneHotEncoder(drop = "first",handle_unknown = "ignore"),["batting_team","bowling_team","city"]),
    ("trf2",PowerTransformer(),["runs_left","balls_left","wickets","total_runs_x","crr","rrr"])
],remainder = "passthrough")

In [7]:
from sklearn.ensemble import GradientBoostingClassifier
gb = Pipeline([
    ("step1",trf),
    ("step2",GradientBoostingClassifier())
])
gb.fit(x_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('step1', ...), ('step2', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['batting_team','bowling_team','city',...,'total_runs_x','crr','rrr']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('trf1', ...), ('trf2', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining

In [8]:
gb_pred = gb.predict(x_test)

In [9]:
df["bowling_team"].unique()

array(['Royal Challengers Bangalore', 'Rajasthan Royals',
       'Chennai Super Kings', 'Kings XI Punjab', 'Mumbai Indians',
       'Kolkata Knight Riders', 'Deccan Chargers', 'Sunrisers Hyderabad',
       'Delhi Daredevils', 'Delhi Capitals'], dtype=object)

In [10]:
df["city"].unique()

array(['Chennai', 'Sharjah', 'Bengaluru', 'Jaipur', 'Mumbai', 'Kimberley',
       'Centurion', 'Kolkata', 'Indore', 'Visakhapatnam', 'Delhi',
       'Cape Town', 'Bangalore', 'Hyderabad', 'Chandigarh', 'Raipur',
       'Durban', 'Ahmedabad', 'Dharamsala', 'Port Elizabeth', 'Abu Dhabi',
       'Ranchi', 'Cuttack', 'Johannesburg', 'Nagpur', 'Pune', 'Mohali',
       'Bloemfontein', 'East London'], dtype=object)

In [11]:
import pickle
pickle.dump(gb,open("gb.pkl","wb"))